In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# 1. Dataset Loading and Initial Integrity Audit
DATA_DIR = Path("../data/raw") # Adjust relative path if running inside /notebooks
COMMODITIES = ['tomato', 'potato', 'onion', 'wheat']

def load_and_inspect_raw_data(data_dir: Path):
    raw_dfs = {}
    print("=" * 80)
    print("PHASE 1: RAW DATASET SCHEMA AND VOLUME AUDIT")
    print("=" * 80)
    
    audit_summary = []
    
    for comm in COMMODITIES:
        file_path = data_dir / f"{comm}.csv"
        if not file_path.exists():
            print(f"[WARNING] File missing: {file_path}")
            continue
            
        df = pd.read_csv(file_path)
        raw_dfs[comm] = df
        
        # Parse date safely
        df['parsed_date'] = pd.to_datetime(df['Reported Date'], errors='coerce')
        invalid_dates = df['parsed_date'].isna().sum()
        
        audit_summary.append({
            'Commodity': comm.capitalize(),
            'Total Rows': len(df),
            'Columns': len(df.columns),
            'Min Date': df['parsed_date'].min().strftime('%Y-%m-%d') if not df['parsed_date'].empty else 'N/A',
            'Max Date': df['parsed_date'].max().strftime('%Y-%m-%d') if not df['parsed_date'].empty else 'N/A',
            'Invalid Dates': invalid_dates,
            'Unique States': df['State Name'].nunique(),
            'Unique Districts': df['District Name'].nunique(),
            'Unique Markets': df['Market Name'].nunique(),
            'Unique Varieties': df['Variety'].nunique()
        })
        
    summary_df = pd.DataFrame(audit_summary)
    print(summary_df.to_string(index=False))
    return raw_dfs

# Execute Phase 1
raw_dfs = load_and_inspect_raw_data(DATA_DIR)

PHASE 1: RAW DATASET SCHEMA AND VOLUME AUDIT
Commodity  Total Rows  Columns   Min Date   Max Date  Invalid Dates  Unique States  Unique Districts  Unique Markets  Unique Varieties
   Tomato     2338352       11 2000-10-19 2024-02-01              0             31               484            1557                 6
   Potato     2764987       11 2000-10-19 2024-02-02              0             33               500            1604                22
    Onion     2624379       11 2000-10-19 2024-02-02              0             32               507            1714                25
    Wheat     2593767       11 2001-03-16 2024-02-01              0             28               452            1956                71


In [2]:
# 2. Granularity, Sparsity, and Gap Analysis
def analyze_granularity_and_gaps(raw_dfs):
    print("\n" + "=" * 80)
    print("PHASE 2: SPATIAL GRANULARITY AND TIME-SERIES CONTINUITY")
    print("=" * 80)
    
    for comm, df in raw_dfs.items():
        df['parsed_date'] = pd.to_datetime(df['Reported Date'], errors='coerce')
        df = df.dropna(subset=['parsed_date'])
        
        # Calculate market vs district density
        market_counts = df.groupby(['District Name', 'Market Name']).size().reset_index(name='record_count')
        avg_records_per_market = market_counts['record_count'].mean()
        
        district_counts = df.groupby('District Name').size().reset_index(name='record_count')
        avg_records_per_district = district_counts['record_count'].mean()
        
        # Test time continuity at top district
        top_district = df['District Name'].value_counts().index[0]
        dist_df = df[df['District Name'] == top_district].sort_values('parsed_date')
        
        full_date_range = pd.date_range(start=dist_df['parsed_date'].min(), end=dist_df['parsed_date'].max())
        unique_dates_present = dist_df['parsed_date'].nunique()
        missing_days = len(full_date_range) - unique_dates_present
        sparsity_pct = (missing_days / len(full_date_range)) * 100
        
        print(f"\n[{comm.upper()}] Granularity & Continuity Insights:")
        print(f"  • Avg Records per Market  : {avg_records_per_market:.1f}")
        print(f"  • Avg Records per District: {avg_records_per_district:.1f}")
        print(f"  • Top District Examined   : '{top_district}'")
        print(f"  • Total Expected Days     : {len(full_date_range)} days")
        print(f"  • Missing Days (Sparsity) : {missing_days} days ({sparsity_pct:.2f}% gap rate)")

# Execute Phase 2
if raw_dfs:
    analyze_granularity_and_gaps(raw_dfs)


PHASE 2: SPATIAL GRANULARITY AND TIME-SERIES CONTINUITY

[TOMATO] Granularity & Continuity Insights:
  • Avg Records per Market  : 1488.4
  • Avg Records per District: 4831.3
  • Top District Examined   : 'Sangrur'
  • Total Expected Days     : 6879 days
  • Missing Days (Sparsity) : 140 days (2.04% gap rate)

[POTATO] Granularity & Continuity Insights:
  • Avg Records per Market  : 1705.7
  • Avg Records per District: 5530.0
  • Top District Examined   : 'Ludhiana'
  • Total Expected Days     : 8156 days
  • Missing Days (Sparsity) : 647 days (7.93% gap rate)

[ONION] Granularity & Continuity Insights:
  • Avg Records per Market  : 1517.9
  • Avg Records per District: 5176.3
  • Top District Examined   : 'Nashik'
  • Total Expected Days     : 7978 days
  • Missing Days (Sparsity) : 1456 days (18.25% gap rate)

[WHEAT] Granularity & Continuity Insights:
  • Avg Records per Market  : 1306.7
  • Avg Records per District: 5738.4
  • Top District Examined   : 'Ganganagar'
  • Total Expect

In [3]:
# 3. Target Variable (Arrivals) & Anomaly Diagnostics
def analyze_arrivals_and_prices(raw_dfs):
    print("\n" + "=" * 80)
    print("PHASE 3: TARGET VARIABLE (ARRIVALS) & PRICE ANOMALIES")
    print("=" * 80)
    
    anomaly_report = []
    
    for comm, df in raw_dfs.items():
        arrivals = df['Arrivals (Tonnes)']
        modal_prices = df['Modal Price (Rs./Quintal)']
        min_prices = df['Min Price (Rs./Quintal)']
        max_prices = df['Max Price (Rs./Quintal)']
        
        # Detect anomaly types
        zero_arrivals = (arrivals <= 0).sum()
        null_arrivals = arrivals.isna().sum()
        p99_arrival = arrivals.quantile(0.99)
        max_arrival = arrivals.max()
        
        # Price anomalies
        zero_prices = (modal_prices <= 0).sum()
        invalid_min_max = (min_prices > max_prices).sum()
        
        anomaly_report.append({
            'Commodity': comm.capitalize(),
            'Null Arrivals': null_arrivals,
            '<= 0 Arrivals': zero_arrivals,
            'Arrivals P50 (Median)': round(arrivals.median(), 2),
            'Arrivals P99': round(p99_arrival, 2),
            'Max Arrival': round(max_arrival, 2),
            '<= 0 Prices': zero_prices,
            'Min > Max Price Bugs': invalid_min_max
        })
        
    anomaly_df = pd.DataFrame(anomaly_report)
    print(anomaly_df.to_string(index=False))

# Execute Phase 3
if raw_dfs:
    analyze_arrivals_and_prices(raw_dfs)


PHASE 3: TARGET VARIABLE (ARRIVALS) & PRICE ANOMALIES
Commodity  Null Arrivals  <= 0 Arrivals  Arrivals P50 (Median)  Arrivals P99  Max Arrival  <= 0 Prices  Min > Max Price Bugs
   Tomato              0             15                    2.5         381.7     210650.0        26740                  3421
   Potato              0              3                   10.0        1158.0     212350.0        32502                  3990
    Onion              0              9                    5.9        1756.0     244420.0        27432                  3282
    Wheat              0             14                   23.2        3318.0    8389090.0        13327                  3436
